# 🚀 Night Loom Engine — Google Colab Free T4 GPU Microservice Worker
This notebook turns a free Google Colab T4 GPU instance into a remote AI microservice endpoint for **SDXL-Turbo Image Generation** and **SadTalker Avatar Video Rendering**.

In [ ]:
# Step 1: Install PyTorch CUDA, Diffusers, FastAPI, Tunneling, and nest_asyncio
!pip install -q diffusers transformers accelerate torch torchvision safetensors pyngrok fastapi uvicorn pydantic python-multipart nest_asyncio pillow

In [ ]:
# Step 2: Clone YT-Automation-Hosted Repository
import os
# If your repo is Private, paste your GitHub Personal Access Token (PAT) below:
GH_TOKEN = ""

if GH_TOKEN:
    clone_url = f"https://{GH_TOKEN}@github.com/1919-14/YT-Automation-Hosted.git"
else:
    clone_url = "https://github.com/1919-14/YT-Automation-Hosted.git"

!git clone {clone_url} /content/YT-Automation-Hosted
%cd /content/YT-Automation-Hosted

In [ ]:
# Step 3: Launch FastAPI GPU Microservice Server with SDXL-Turbo Endpoint
import os
import sys
import time
import base64
import threading
from io import BytesIO
import torch
import uvicorn
import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
from diffusers import AutoPipelineForText2Image

# Apply patch for Jupyter notebook asyncio loop
nest_asyncio.apply()

# 🔑 Paste your free ngrok Authtoken below (Get it from https://dashboard.ngrok.com/get-started/your-authtoken):
NGROK_AUTHTOKEN = ""

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

app = FastAPI(title="Night Loom GPU Worker API")

# Global SDXL-Turbo Pipeline lazy loader
_sdxl_pipe = None

def get_sdxl_pipe():
    global _sdxl_pipe
    if _sdxl_pipe is None:
        print("[Colab GPU] Loading SDXL-Turbo model into VRAM...")
        _sdxl_pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/sdxl-turbo",
            torch_dtype=torch.float16,
            variant="fp16"
        ).to("cuda")
        print("[Colab GPU] SDXL-Turbo ready on CUDA!")
    return _sdxl_pipe

class ImageGenRequest(BaseModel):
    prompts: list[str]
    width: int = 576
    height: int = 1024
    num_inference_steps: int = 4

@app.get("/health")
def healthcheck():
    return {
        "status": "online",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
        "vram_free_gb": round(torch.cuda.mem_get_info()[0] / 1024**3, 2) if torch.cuda.is_available() else 0
    }

@app.post("/generate_images")
def generate_images(req: ImageGenRequest):
    try:
        pipe = get_sdxl_pipe()
        encoded_images = []
        print(f"[Colab GPU] Generating {len(req.prompts)} image(s)...")
        for i, prompt in enumerate(req.prompts):
            image = pipe(
                prompt=prompt,
                num_inference_steps=req.num_inference_steps,
                guidance_scale=0.0,
                width=req.width,
                height=req.height
            ).images[0]
            buf = BytesIO()
            image.save(buf, format="PNG")
            b64_str = base64.b64encode(buf.getvalue()).decode("utf-8")
            encoded_images.append(b64_str)
        return {"status": "success", "images": encoded_images}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

PORT = 8000
if not NGROK_AUTHTOKEN:
    print("❌ NGROK_AUTHTOKEN IS MISSING!")
    print("👉 Get your free token in 10 seconds here: https://dashboard.ngrok.com/get-started/your-authtoken")
    print("👉 Paste it into NGROK_AUTHTOKEN = 'your_token_here' above and re-run cell 3.")
else:
    public_url = ngrok.connect(PORT).public_url
    print("\n" + "="*60)
    print("⚡ GPU WORKER IS ONLINE!")
    print("🔑 Copy this GPU_WORKER_URL into your Hugging Face Space Secrets:")
    print(f"👉 GPU_WORKER_URL = {public_url}")
    print("="*60 + "\n")
    
    # Launch uvicorn server in a background thread to prevent Jupyter asyncio clashes
    config = uvicorn.Config(app=app, host="0.0.0.0", port=PORT)
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    print("✅ FastAPI Server is running in background thread!")
